# Block 2 — Scene Segmentation and Character Identification

This notebook walks from a raw screenplay file to a confirmed `characters.csv` entry for one film. It is **film-agnostic**: the only cell that changes between films is the parameters cell at the top.

## Workflow structure

| Phase | Sections | What happens |
|---|---|---|
| **Phase 1 — Parse** | §1 – §4 | Load film metadata → normalize text → segment scenes → parse all speaker cues |
| **Phase 2 — Character review** | §5 | Save character candidates and review template — **stop here and fill in the template** |
| **Manual review** | — | Open `character_review_{film_id}.csv`, fill in `gender` / `is_named` / `is_protagonist` / `included_in_network` |
| **Phase 3 — Commit** | §6 – §8 | Load reviewed template → validate → filter and save utterances → build Block 2 schema → append to master |

## Files produced

```
CLEAN/data/processed/{film_id}/
    character_candidates.csv          — raw parser output, one row per unique speaker (§5)
    character_review_{film_id}.csv    — you fill this in during the manual review step (§5)
    scenes.csv                        — one row per scene, saved after review (§7)
    utterances.csv                    — one row per speech turn, named characters only (§7)

CLEAN/data/00_corpus/
    characters.csv                    — master file, one row per (film, character), accumulated across all films
```

## Design notes

All parsing logic lives in `CLEAN/scripts/screenplay_parser_utils.py`. This notebook orchestrates those helpers, provides the inspection and annotation layer, and writes outputs. No parsing logic is duplicated here.

---
## Parameters

**Change only this cell between films.** Everything below resolves from `final_films.csv` using `film_id`.

In [1]:
# ── CHANGE ONLY THESE TWO LINES FOR EACH FILM ─────────────────────────────────
film_id = "elemental_2023"

# Set True only for films whose raw file contains the full screenplay twice.
# Currently only needed for: beautyandthebeast
trim_duplicate = True
# ──────────────────────────────────────────────────────────────────────────────

## Setup — imports and path resolution

The `_find_clean_root` helper walks up from the current working directory looking for the `CLEAN/` folder (identified by its `admin/` and `data/` children). This makes the notebook work regardless of where Jupyter was launched from.

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def _find_clean_root() -> Path:
    """Locate the CLEAN project root regardless of Jupyter launch directory."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if candidate.name == "CLEAN" and (candidate / "admin").exists() and (candidate / "data").exists():
            return candidate
        clean_child = candidate / "CLEAN"
        if clean_child.is_dir() and (clean_child / "admin").exists() and (clean_child / "data").exists():
            return clean_child
    raise FileNotFoundError(
        f"Cannot locate CLEAN root from {here}. "
        "Run this notebook from inside the CLEAN directory tree."
    )


CLEAN_ROOT = _find_clean_root()
SCRIPTS_DIR = CLEAN_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from screenplay_parser_utils import (
    apply_reviewed_characters_to_utterances,
    build_character_candidates_dataframe,
    build_character_review_template,
    build_scenes_dataframe,
    build_utterances_dataframe,
    clean_script_text,
    load_film_metadata,
    parse_dialogue_within_scene,
    segment_scenes,
    trim_repeated_screenplay,
)

FILMS_CSV    = CLEAN_ROOT / "data" / "00_corpus" / "final_films.csv"
OUTPUT_DIR   = CLEAN_ROOT / "data" / "02_processed" / film_id
METADATA_DIR = CLEAN_ROOT / "data" / "00_corpus"
MASTER_CHARS = METADATA_DIR / "characters.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"CLEAN root   : {CLEAN_ROOT}")
print(f"Film ID      : {film_id}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Master chars : {MASTER_CHARS}")

CLEAN root   : C:\Users\majoa\OneDrive\Desktop\Learning\Grad_School\final_master_thesis\CLEAN
Film ID      : elemental_2023
Output dir   : C:\Users\majoa\OneDrive\Desktop\Learning\Grad_School\final_master_thesis\CLEAN\data\processed\elemental_2023
Master chars : C:\Users\majoa\OneDrive\Desktop\Learning\Grad_School\final_master_thesis\CLEAN\data\metadata\characters.csv


---
## §1 — Film Metadata

`final_films.csv` holds one row per film with the parser configuration fields. The key fields for this notebook:

| Field | What it controls |
|---|---|
| `marker_type` | Which scene-boundary strategy to use (`slug_line`, `scene_header`, `separator_line`, `numbered_section`) |
| `scene_marker_regex` | The regex that identifies a scene-boundary line within the normalized text |
| `script_template` | Speaker-cue format (`indented_screenplay`, `inline_colon_mixed`, `inline_colon_upper`) |
| `min_speaker_indent` | Minimum left-indent (spaces) for a line to be treated as a speaker cue (only for `indented_screenplay`) |
| `min_dialogue_indent` | Minimum left-indent for a line to be treated as dialogue continuation (only for `indented_screenplay`) |

**Check:** Confirm that `marker_type` and `script_template` look correct for this film before running the parser.

In [3]:
film = load_film_metadata(FILMS_CSV, film_id)

# Display as a clean two-column table
meta_df = pd.Series(film).rename("value").to_frame()
display(meta_df)

,value
movie_id,elemental_2023
movie_name,Elemental
year_released,2023
lead_gender,female
script_name,elemental_2023.txt
marker_type,slug_line
scene_marker_regex,^\s*(?:INT\.|EXT\.)\s+.+$
notes,Indented screenplay slug lines; inline-colon p...
studio,pixar
script_path,data/raw_scripts/elemental_2023.txt


---
## §2 — Read and Normalize the Screenplay

The raw screenplay text is read and passed through two normalization steps:

1. **`clean_script_text`** — normalizes Unicode (smart quotes, em-dashes, encoding artifacts like `â€™`), strips bare page numbers and title-header echoes, and collapses runs of more than two consecutive blank lines. This is a conservative pass: it does not rewrite dialogue content.

2. **`trim_repeated_screenplay`** (conditional) — detects and removes a duplicated copy of the screenplay body. Currently needed for the Beauty and the Beast source file, which contains the script twice. The function matches on the first `Scene 1` occurrence and cuts everything before the second one.

**Commentary space:** Inspect the first 1 500 characters of the normalized text. Confirm that the scene marker pattern (`scene_marker_regex`) is visible in this opening block.

In [4]:
script_path = CLEAN_ROOT / str(film["script_path"])
raw_text = script_path.read_text(encoding="utf-8", errors="ignore")

print(f"Script path : {script_path}")
print(f"Raw chars   : {len(raw_text):,}")
print(f"Raw lines   : {len(raw_text.splitlines()):,}")

title_header = str(film.get("movie_name", "")).upper().strip() or None
cleaned_text = clean_script_text(raw_text, title_header=title_header)

if trim_duplicate:
    trimmed_text, trim_audit = trim_repeated_screenplay(cleaned_text)
    print(f"\nDuplicate trim: {trim_audit}")
else:
    trimmed_text = cleaned_text

print(f"\nFinal line count: {len(trimmed_text.splitlines()):,}")
print("\n" + "─" * 60)
print(trimmed_text[:1500])

Script path : C:\Users\majoa\OneDrive\Desktop\Learning\Grad_School\final_master_thesis\CLEAN\data\raw_scripts\elemental_2023.txt
Raw chars   : 131,882
Raw lines   : 5,066

Duplicate trim: {'duplicate_trimmed': False, 'repeat_start_line': None, 'original_line_count': 4817, 'trimmed_line_count': 4817}

Final line count: 4,817

────────────────────────────────────────────────────────────
Original Story by
Peter Sohn
John Hoberg & Kat Likkel
Brenda Hsueh
Screenplay by
John Hoberg & Kat Likkel
Brenda Hsueh
Pixar Animation Studios
1200 Park Avenue
Emeryville, CA 94608

EXT. OCEAN - DAY
Through the mist we see a glow. A small weathered boat breaks
through, traveling toward camera. At first it appears the
deck is on fire, but as the boat gets closer, we see the
“fire” is actually two FIRE ELEMENTS, BERNIE and a very
pregnant CINDER.
TITLE: DISNEY PRESENTS
They huddle together surrounded by suitcases and a small
lantern that holds a BLUE FLAME.
Bernie opens one of the suitcases, takes out a met

---
## §3 — Scene Segmentation

`segment_scenes` slices the normalized script at every line matching the film's `scene_marker_regex`, subject to additional sanity checks per `marker_type`:

- **`separator_line`** (Finding Nemo): requires ≥ 5 `=` characters AND a blank line immediately above and below — prevents cosmetic underlines from being treated as scene breaks.
- **`numbered_section`** (Inside Out): requires a two-digit number followed by all-caps text — prevents dates and address lines from matching.
- **`scene_header`** (Beauty and the Beast, Princess and the Frog, Monsters Inc, Incredibles 2): requires the `Scene ` prefix.
- **`slug_line`** (most films): matched directly by the regex.

Each `SceneSegment` carries a stable `scene_id` (`{film_id}_s{N:03d}`), the raw header line, and the block of text between this boundary and the next.

**Commentary space:** Check the scene count (typical range: 30–80 scenes). Note any headers that look like false positives or that are unusually short.

In [5]:
scene_segments = segment_scenes(
    text=trimmed_text,
    film_id=film_id,
    marker_type=str(film["marker_type"]),
    scene_marker_regex=str(film["scene_marker_regex"]),
)

scene_preview_df = pd.DataFrame([
    {
        "scene_id": s.scene_id,
        "scene_number": s.scene_number,
        "scene_header": s.scene_header,
        "start_line": s.start_line,
        "end_line": s.end_line,
        "block_lines": s.end_line - s.start_line + 1,
    }
    for s in scene_segments
])

print(f"Total scenes segmented: {len(scene_segments)}")
display(scene_preview_df)

Total scenes segmented: 121


,scene_id,scene_number,scene_header,start_line,end_line,block_lines
0,elemental_2023_s001,1,EXT. OCEAN - DAY,12,31,20
1,elemental_2023_s002,2,EXT. GRAND GATEWAY DOCKS - DAY,32,62,31
2,elemental_2023_s003,3,INT. IMMIGRATION HALL - DAY - CONTINUOUS,63,92,30
3,elemental_2023_s004,4,EXT. ELEMENT CITY - STREET - SHORTLY AFTER,93,106,14
4,elemental_2023_s005,5,INT. TRAIN - SHORTLY AFTER,107,119,13
...,...,...,...,...,...,...
116,elemental_2023_s117,117,EXT. FIRETOWN STREETS - MONTHS LATER,4683,4687,5
117,elemental_2023_s118,118,INT. BERNIE’S SHOP - CONTINUOUS,4688,4731,44
118,elemental_2023_s119,119,EXT. BERNIE’S SHOP - SAME TIME,4732,4735,4
119,elemental_2023_s120,120,INT. BERNIE'S SHOP - CONTINUOUS,4736,4763,28


---
## §4 — Dialogue Parsing and Character Candidates

For each scene, `parse_dialogue_within_scene` detects speaker-cue lines and collects the dialogue lines that follow. The detection strategy is determined by `script_template`:

- **`indented_screenplay`** — looks for all-uppercase lines with left-indent ≥ `min_speaker_indent`.
- **`inline_colon_mixed`** / **`inline_colon_upper`** — looks for `Speaker: dialogue` patterns on a single line; also handles bracketed `[SPEAKER]` cues (common in song sections).

The three output DataFrames:

| DataFrame | Rows | Key use |
|---|---|---|
| `utterances_df` | One per speech turn | Join key for sentiment, discourse, network (Blocks 4–6) |
| `scenes_df` | One per scene | Scene-level summary with utterance counts |
| `character_candidates_df` | One per unique speaker | Input for Block 2 manual review |

**Commentary space:** Inspect the character candidates table. Check for:
- **Apparent duplicates** — e.g., `BELLE` and `BELLE (V.O.)` as separate rows. Flag these in `merge_into_character_id` during review.
- **Non-character labels** that slipped through — stage directions (`CHORUS`, `ALL`) or formatting artifacts. Set `included_in_network = False` for these.
- **Characters with very few utterances** — the network analysis threshold is 5 utterances; set `included_in_network = False` for those below it.

In [6]:
parsed_utterances = []
for scene in scene_segments:
    parsed_utterances.extend(
        parse_dialogue_within_scene(
            scene=scene,
            script_template=str(film["script_template"]),
            min_speaker_indent=int(film["min_speaker_indent"] or 0),
            min_dialogue_indent=int(film["min_dialogue_indent"] or 0),
        )
    )

utterances_df        = build_utterances_dataframe(parsed_utterances, film_id=film_id)
scenes_df            = build_scenes_dataframe(scene_segments, utterances_df)
character_candidates_df = build_character_candidates_dataframe(utterances_df)

print(f"Utterances parsed    : {len(utterances_df)}")
print(f"  — spoken           : {int((utterances_df['utterance_type'] == 'spoken').sum())}")
print(f"  — song             : {int((utterances_df['utterance_type'] == 'song').sum())}")
print(f"Scenes retained      : {len(scenes_df)}")
print(f"Character candidates : {len(character_candidates_df)}")

Utterances parsed    : 942
  — spoken           : 937
  — song             : 5
Scenes retained      : 121
Character candidates : 65


In [7]:
# Scene summary: how many utterances per scene?
display(
    scenes_df[[
        "scene_id", "scene_number", "scene_header",
        "utterance_count", "spoken_utterance_count", "song_utterance_count",
    ]]
)

,scene_id,scene_number,scene_header,utterance_count,spoken_utterance_count,song_utterance_count
0,elemental_2023_s001,1,EXT. OCEAN - DAY,1,1,0
1,elemental_2023_s002,2,EXT. GRAND GATEWAY DOCKS - DAY,3,3,0
2,elemental_2023_s003,3,INT. IMMIGRATION HALL - DAY - CONTINUOUS,7,7,0
3,elemental_2023_s004,4,EXT. ELEMENT CITY - STREET - SHORTLY AFTER,3,3,0
4,elemental_2023_s005,5,INT. TRAIN - SHORTLY AFTER,1,1,0
...,...,...,...,...,...,...
116,elemental_2023_s117,117,EXT. FIRETOWN STREETS - MONTHS LATER,1,1,0
117,elemental_2023_s118,118,INT. BERNIE’S SHOP - CONTINUOUS,9,9,0
118,elemental_2023_s119,119,EXT. BERNIE’S SHOP - SAME TIME,0,0,0
119,elemental_2023_s120,120,INT. BERNIE'S SHOP - CONTINUOUS,8,8,0


In [8]:
# Character candidates sorted by utterance count (most lines first)
# This is the raw parser output — gender and protagonist status are blank until manual review.
display(
    character_candidates_df[[
        "canonical_name",
        "raw_speaker_labels",
        "raw_speaker_label_variant_count",
        "utterance_count",
        "spoken_utterance_count",
        "song_utterance_count",
        "word_count_spoken_only",
        "word_count_song_only",
    ]]
)

,canonical_name,raw_speaker_labels,raw_speaker_label_variant_count,utterance_count,spoken_utterance_count,song_utterance_count,word_count_spoken_only,word_count_song_only
24,Ember,EMBER | EMBER (O.S.) | EMBER (V.O.),3,316,314,2,5875,75
61,Wade,WADE | WADE (CONT’D) | WADE (O.S.) | WADE (V.O.),4,253,253,0,4508,0
5,Bernie,BERNIE | BERNIE (CONT’D) | BERNIE (O.S.) | BER...,4,127,127,0,2137,0
9,Cinder,CINDER | CINDER (CONT'D) | CINDER (O.S.) | CIN...,5,57,57,0,971,0
34,Gale,GALE,1,28,28,0,469,0
...,...,...,...,...,...,...,...,...
55,This Stop City Hall.,THIS STOP CITY HALL.,1,1,1,0,14,0
57,Title Over Black,TITLE OVER BLACK,1,1,1,0,20,0
58,Toot Toot Juice Vendor,TOOT TOOT JUICE VENDOR,1,1,1,0,26,0
59,Train Announcer,TRAIN ANNOUNCER (O.S.),1,1,1,0,43,0


### Utterance sample

A random sample of 20 utterances is shown below to spot-check parser output quality: speaker labels, text cleanliness, and song tagging.

In [9]:
sample = utterances_df.sample(n=min(20, len(utterances_df)), random_state=42)
display(
    sample[[
        "event_id", "scene_id", "speaker", "text", "utterance_type", "word_count"
    ]].sort_values("event_id")
)

,event_id,scene_id,speaker,text,utterance_type,word_count
23,24,elemental_2023_s009,CINDER,Íkì ss ûr.,spoken,3
30,31,elemental_2023_s010,BERNIE,Hnnng. ERgggg. Hmmmm! -- The shop is mid-const...,spoken,37
63,64,elemental_2023_s016,BERNIE,How about... You take it today. Ember’s eyes g...,spoken,14
107,108,elemental_2023_s017,BERNIE,Just tired.,spoken,2
158,159,elemental_2023_s024,EMBER,"Dad, I'll take care of it. You need to rest. J...",spoken,25
215,216,elemental_2023_s028,WADE,Sorry! I gotta get these to city hall before t...,spoken,42
261,262,elemental_2023_s037,WADE,Sup--,spoken,1
334,335,elemental_2023_s045,EMBER,Thirty?? Wade shrugs apologetically. Ember tur...,spoken,9
361,362,elemental_2023_s045,WADE,"Uh! Yeah! Yeah! Yes! Ember looks at Wade, amaz...",spoken,14
371,372,elemental_2023_s046,GALE,WATER? In FIRETOWN?,spoken,3


---
## §5 — Save Character Candidates for Review

Two files are written to `data/processed/{film_id}/`. **`scenes.csv` and `utterances.csv` are saved in §7, after you have reviewed the characters.**

1. **`character_candidates.csv`** — aggregated speaker stats, direct from the parser.
2. **`character_review_{film_id}.csv`** — the candidates with six blank annotation columns: `gender`, `gender_rationale`, `is_named`, `is_protagonist`, `included_in_network`, `review_notes`. **This is the file you open for manual review.**

In [ ]:
# candidates_path  = OUTPUT_DIR / "character_candidates.csv"
# review_tmpl_path = OUTPUT_DIR / f"character_review_{film_id}.csv"

# character_candidates_df.to_csv(candidates_path, index=False)

# review_template_df = build_character_review_template(character_candidates_df)
# review_template_df.to_csv(review_tmpl_path, index=False)

# print(f"character_candidates.csv           → {candidates_path}")
# print(f"character_review_{film_id}.csv     → {review_tmpl_path}")
# print(f"\nReview template: {len(review_template_df)} rows × {len(review_template_df.columns)} columns")
# print(f"\n⚠  scenes.csv and utterances.csv are saved in §7, after character review.")

character_candidates.csv           → C:\Users\majoa\OneDrive\Desktop\Learning\Grad_School\final_master_thesis\CLEAN\data\processed\mulan_1998\character_candidates.csv
character_review_mulan_1998.csv     → C:\Users\majoa\OneDrive\Desktop\Learning\Grad_School\final_master_thesis\CLEAN\data\processed\mulan_1998\character_review_mulan_1998.csv

Review template: 50 rows × 22 columns

⚠  scenes.csv and utterances.csv are saved in §7, after character review.


---

# ⬇ MANUAL REVIEW — STOP HERE ⬇

Open the review template file in Excel or Google Sheets (path printed above), fill in the six required columns for **every row you keep**, and save it back as CSV with the same filename before continuing.

---

### Required columns (no blanks allowed, for rows you keep)

| Column | Allowed values | Notes |
|---|---|---|
| `gender` | `F` or `M` | Assign narrative gender. Nonhuman characters get their narrative gender (Beast = M, Olaf = M). No `NHuman` category — see execution plan §1. |
| `is_named` | `True` or `False` | Named character (Belle, Gaston) vs. unnamed (GUARD, VILLAGER 1). Utterances from `False` rows are **excluded** from `utterances.csv`. |
| `is_protagonist` | `True` or `False` | `True` for the film's protagonist(s) only; `False` for everyone else |
| `included_in_network` | `True` or `False` | `False` for any character with < 5 utterances, or for labels that are not real speaking characters |

### Optional columns

| Column | When to fill it |
|---|---|
| `gender_rationale` | One-line note only when gender is nonhuman or ambiguous (e.g., *"Beast is animal-form male character"*). Leave blank for unambiguous cases. |
| `canonical_name` | Correct only if the parser's inferred name is wrong or inconsistent with the film (e.g., `Mrs Potts` → `Mrs. Potts`). |
| `review_character_id` | Correct only if the parser's inferred ID is wrong. Use `lower_snake_case` (e.g., `the_beast`, `mrs_potts`). |
| `merge_into_character_id` | Fill with the **target** `review_character_id` if this row is an alias of another character and should be collapsed into it. Example: a `BELLE (V.O.)` row should have `merge_into_character_id = beautyandthebeast_belle`. Leave blank otherwise. |
| `review_notes` | Free-text annotation — flag anything unusual for team discussion. |

---

### Removing bad parses

If a row is a parser artifact — a sentence fragment, a stage direction, or any label that is not a real speaker — **delete the entire row** from the CSV. All utterances attributed to that label will be dropped from `utterances.csv` in §7. You do not need to fill in any fields for deleted rows.

Use `is_named = False` (instead of deleting) when the label is a real collective speaker (e.g., CROWD, ALL) that you want to keep in the candidates file for reference but exclude from utterances.

---

**When done:** Save the CSV (same filename `character_review_{film_id}.csv`, same location). Then run §6 → §8 below.

In [ ]:
# # Convenience: print the exact file path to open
# print("File to open for manual review:")
# print(f"  {review_tmpl_path.resolve()}")

File to open for manual review:


NameError: name 'review_tmpl_path' is not defined

---
## §6 — Load and Validate the Reviewed Template

This section reads back the filled-in CSV and runs validation checks on every required field. All issues are collected and reported together so you can fix the template in one pass.

Fix any `❌` issues in the template file, save, and re-run this cell before moving to §7.

In [11]:
review_tmpl_path = OUTPUT_DIR / f"character_review_{film_id}.csv"

if not review_tmpl_path.exists():
    raise FileNotFoundError(
        f"Review template not found at {review_tmpl_path}.\n"
        "Run §1–§5 first to generate it, then fill it in and save."
    )

reviewed_df = pd.read_csv(review_tmpl_path, dtype=str).fillna("")

print(f"Loaded {len(reviewed_df)} rows from: {review_tmpl_path}")
print()

display(
    reviewed_df[[
        "canonical_name",
        "gender",
        "gender_rationale",
        "is_named",
        "is_protagonist",
        "included_in_network",
        "merge_into_character_id",
        "utterance_count",
        "review_notes",
    ]]
)

Loaded 52 rows from: C:\Users\majoa\OneDrive\Desktop\Learning\Grad_School\final_master_thesis\CLEAN\data\processed\elemental_2023\character_review_elemental_2023.csv



,canonical_name,gender,gender_rationale,is_named,is_protagonist,included_in_network,merge_into_character_id,utterance_count,review_notes
0,Ember,F,,1,1,1,,316,
1,Wade,M,,1,0,1,,253,
2,Bernie,M,,1,0,1,,128,
3,Cinder,F,,1,0,1,,57,
4,Brook,F,,1,0,1,,19,
5,Gale,F,,1,0,1,,28,
6,Harold,M,,1,0,1,,9,
7,Clod,M,,1,0,1,,14,
8,Earth Kids.,Both,,0,0,1,,1,
9,Teenage Ember,F,,1,1,1,elemental_2023_ember,10,


In [12]:
VALID_GENDERS = {"F", "M", "Unknown", "Both"}
VALID_BOOLS   = {"True", "False", "true", "false", "TRUE", "FALSE", "1", "0"}

errors = []

for idx, row in reviewed_df.iterrows():
    name = str(row.get("canonical_name", f"row {idx}")).strip() or f"row {idx}"

    # Skip alias rows — they don't need gender/protagonist/network since they collapse into the target
    is_alias = str(row.get("merge_into_character_id", "")).strip() != ""
    if is_alias:
        continue

    if str(row.get("gender", "")).strip() not in VALID_GENDERS:
        errors.append(f"  Row {idx} ({name}): 'gender' must be F or M — got '{row.get('gender')}'")

    if str(row.get("is_named", "")).strip() not in VALID_BOOLS:
        errors.append(f"  Row {idx} ({name}): 'is_named' must be True/False — got '{row.get('is_named')}'")

    if str(row.get("is_protagonist", "")).strip() not in VALID_BOOLS:
        errors.append(f"  Row {idx} ({name}): 'is_protagonist' must be True/False — got '{row.get('is_protagonist')}'")

    if str(row.get("included_in_network", "")).strip() not in VALID_BOOLS:
        errors.append(f"  Row {idx} ({name}): 'included_in_network' must be True/False — got '{row.get('included_in_network')}'")

if errors:
    print(f"❌ Validation failed — {len(errors)} issue(s):\n")
    for err in errors:
        print(err)
    raise ValueError("Fix the review template before running §7.")

# Summary
canonical_rows = reviewed_df[reviewed_df["merge_into_character_id"].str.strip() == ""]
protagonist_rows = canonical_rows[
    canonical_rows["is_protagonist"].str.strip().str.lower().isin({"true", "1"})
]
network_included = canonical_rows[
    canonical_rows["included_in_network"].str.strip().str.lower().isin({"true", "1"})
]

print("✅ Validation passed\n")
print(f"  Canonical characters : {len(canonical_rows)}")
print(f"  Alias rows (merges)  : {len(reviewed_df) - len(canonical_rows)}")
print(f"  Protagonist(s)       : {', '.join(protagonist_rows['canonical_name'].tolist())}")
print(f"  Included in network  : {len(network_included)}")
print(f"  Female / Male        : "
      f"{(canonical_rows['gender'].str.upper() == 'F').sum()} F / "
      f"{(canonical_rows['gender'].str.upper() == 'M').sum()} M")

✅ Validation passed

  Canonical characters : 49
  Alias rows (merges)  : 3
  Protagonist(s)       : Ember
  Included in network  : 15
  Female / Male        : 16 F / 24 M


---
## §7 — Apply Reviewed Characters and Save Utterances

Utterances are filtered in two passes before being saved:

| Pass | Mechanism | Effect |
|---|---|---|
| **1 — Deleted rows** | Character's row is absent from the review template | All utterances for that speaker are silently dropped |
| **2 — `is_named = False`** | Row is present but marked not-named (GUARD, TOWNSPEOPLE, song fragments) | Utterances dropped |

Remaining utterances are enriched with your reviewed `canonical_name`, `character_id`, and metadata, then saved to `utterances.csv`.

In [13]:
# Pass 1 — drop utterances for characters deleted from the review template.
# (Pass 2 / is_named filter was removed: is_named is now enriched onto each row
# but utterances for unnamed speakers are preserved. Downstream consumers
# filter on `is_named` or `included_in_network` themselves.)
reviewed_ids = set(reviewed_df["parser_character_id"].astype(str).str.strip())
utterances_in_scope = utterances_df[
    utterances_df["character_id"].astype(str).isin(reviewed_ids)
].copy()
n_deleted_chars = utterances_df["character_id"].nunique() - utterances_in_scope["character_id"].nunique()
n_deleted_utts  = len(utterances_df) - len(utterances_in_scope)
if n_deleted_chars:
    print(f"Pass 1 — deleted from template : {n_deleted_chars} characters, {n_deleted_utts} utterances dropped")

# Rename review_character_id → character_id so apply_reviewed_characters_to_utterances can match
reviewed_for_utterances = reviewed_df.rename(columns={"review_character_id": "character_id"})

# Apply reviewed metadata (updates character_id, canonical_name, adds gender, is_named, etc.)
utterances_final = apply_reviewed_characters_to_utterances(
    utterances_df=utterances_in_scope,
    reviewed_characters_df=reviewed_for_utterances,
).reset_index(drop=True)

n_named   = int(utterances_final["is_named"].astype(str).str.strip().str.lower().isin({"true", "1"}).sum())
n_unnamed = len(utterances_final) - n_named

# Save
scenes_path     = OUTPUT_DIR / "scenes.csv"
utterances_path = OUTPUT_DIR / "utterances.csv"
scenes_df.to_csv(scenes_path, index=False)
utterances_final.to_csv(utterances_path, index=False)

print(f"\nFinal utterances : {len(utterances_final)}")
print(f"  — spoken       : {int((utterances_final['utterance_type'] == 'spoken').sum())}")
print(f"  — song         : {int((utterances_final['utterance_type'] == 'song').sum())}")
print(f"  — named        : {n_named}")
print(f"  — unnamed      : {n_unnamed}  (kept for data preservation; filter downstream)")
print(f"\nscenes.csv     → {scenes_path}")
print(f"utterances.csv → {utterances_path}")

Pass 1 — deleted from template : 13 characters, 16 utterances dropped

Final utterances : 926
  — spoken       : 921
  — song         : 5
  — named        : 868
  — unnamed      : 58  (kept for data preservation; filter downstream)

scenes.csv     → C:\Users\majoa\OneDrive\Desktop\Learning\Grad_School\final_master_thesis\CLEAN\data\processed\elemental_2023\scenes.csv
utterances.csv → C:\Users\majoa\OneDrive\Desktop\Learning\Grad_School\final_master_thesis\CLEAN\data\processed\elemental_2023\utterances.csv


---
## §8 — Build Block 2 Characters Schema and Commit to Master

### Character schema (Block 2 of the execution plan)

```
film_id               str
character_id          str   stable lower_snake_case identifier
canonical_name        str   display name (e.g., "Belle", "The Beast")
raw_speaker_labels    str   pipe-separated list of all raw variants seen in the script
gender                str   F | M
gender_rationale      str   one-line note for nonhuman or ambiguous cases; blank otherwise
is_protagonist        bool
is_named              bool
included_in_network   bool  False for characters with < 5 utterances or non-character labels
utterance_count       int   total utterances from the parser
word_count_spoken_only int  spoken words only (excludes song lyrics)
word_count_song_only  int   song lyric words only
notes                 str   optional free text
```

### Merge handling

Any row with a non-empty `merge_into_character_id` is treated as an alias and collapsed into the target character. Its utterance and word counts are added to the target row's totals, and its `raw_speaker_labels` variants are folded into the target's label list. The alias row itself does not appear in the final output.

### Re-run safety

If you re-run this cell for a film that has already been committed to `characters.csv`, the existing rows for that film are replaced rather than duplicated.

In [14]:
def _coerce_bool(value: str) -> bool:
    return str(value).strip().lower() in {"true", "1", "yes"}


def build_characters_csv_rows(reviewed_df: pd.DataFrame, film_id: str) -> pd.DataFrame:
    """Convert the filled-in review template into the Block 2 characters schema."""
    df = reviewed_df.copy()

    # Normalize all string columns
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

    # Numeric columns (may come back as strings from CSV)
    count_cols = [
        "utterance_count", "spoken_utterance_count", "song_utterance_count",
        "word_count_total", "word_count_spoken_only", "word_count_song_only",
    ]
    for col in count_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    # Split alias rows from canonical rows
    is_alias   = df["merge_into_character_id"].str.strip().ne("")
    aliases    = df[is_alias].copy()
    canonical  = df[~is_alias].copy().reset_index(drop=True)

    # Fold alias counts and labels into their target canonical row
    if not aliases.empty:
        active_count_cols = [c for c in count_cols if c in canonical.columns]
        for _, alias_row in aliases.iterrows():
            target_id = alias_row["merge_into_character_id"]
            mask = canonical["review_character_id"] == target_id
            if not mask.any():
                print(f"  Warning: merge_into_character_id '{target_id}' not found — skipping alias '{alias_row['canonical_name']}'")
                continue
            for col in active_count_cols:
                canonical.loc[mask, col] += alias_row.get(col, 0)
            # Merge raw_speaker_labels
            existing = canonical.loc[mask, "raw_speaker_labels"].iloc[0]
            combined = " | ".join(sorted({
                lbl.strip()
                for lbl in (existing + " | " + alias_row.get("raw_speaker_labels", "")).split("|")
                if lbl.strip()
            }))
            canonical.loc[mask, "raw_speaker_labels"] = combined

    # Build output rows
    out_rows = []
    for _, row in canonical.iterrows():
        out_rows.append({
            "film_id":              film_id,
            "character_id":         row["review_character_id"],
            "canonical_name":       row["canonical_name"],
            "raw_speaker_labels":   row["raw_speaker_labels"],
            "gender":               row["gender"],
            "gender_rationale":     row.get("gender_rationale", ""),
            "is_protagonist":       _coerce_bool(row["is_protagonist"]),
            "is_named":             _coerce_bool(row["is_named"]),
            "included_in_network":  _coerce_bool(row["included_in_network"]),
            "utterance_count":      int(row.get("utterance_count", 0)),
            "word_count_spoken_only": int(row.get("word_count_spoken_only", 0)),
            "word_count_song_only": int(row.get("word_count_song_only", 0)),
            "notes":                row.get("review_notes", ""),
        })

    return pd.DataFrame(out_rows)


film_characters_df = build_characters_csv_rows(reviewed_df, film_id)

print(f"Built {len(film_characters_df)} character rows for '{film_id}'")
print()
display(film_characters_df)

Built 49 character rows for 'elemental_2023'



,film_id,character_id,canonical_name,raw_speaker_labels,gender,gender_rationale,is_protagonist,is_named,included_in_network,utterance_count,word_count_spoken_only,word_count_song_only,notes
0,elemental_2023,elemental_2023_ember,Ember,BIG KID EMBER | EMBER | EMBER (O.S.) | EMBER (...,F,,True,True,True,336,6305,90,
1,elemental_2023,elemental_2023_wade,Wade,WADE | WADE (CONT’D) | WADE (O.S.) | WADE (V.O.),M,,False,True,True,253,4566,0,
2,elemental_2023,elemental_2023_bernie,Bernie,BERNIE | BERNIE (CONT’D) | BERNIE (O.S.) | BER...,M,,False,True,True,128,2155,0,
3,elemental_2023,elemental_2023_cinder,Cinder,CINDER | CINDER (CONT'D) | CINDER (O.S.) | CIN...,F,,False,True,True,57,979,0,
4,elemental_2023,elemental_2023_brook,Brook,BROOK | BROOK (CONT’D),F,,False,True,True,19,442,49,
5,elemental_2023,elemental_2023_gale,Gale,GALE,F,,False,True,True,28,479,0,
6,elemental_2023,elemental_2023_harold,Harold,HAROLD,M,,False,True,True,9,218,0,
7,elemental_2023,elemental_2023_clod,Clod,CLOD,M,,False,True,True,14,197,0,
8,elemental_2023,elemental_2023_earth_kids,Earth Kids.,EARTH KIDS.,Both,,False,False,True,1,172,0,
9,elemental_2023,elemental_2023_water_passenger,Water Passenger,WATER PASSENGER 1 | WATER PASSENGER 2,F,,False,False,False,2,90,0,


In [15]:
# Load existing master or start fresh
if MASTER_CHARS.exists():
    master_df = pd.read_csv(MASTER_CHARS, dtype=str)
    # Drop existing rows for this film so re-runs are idempotent
    before_count = len(master_df)
    master_df = master_df[master_df["film_id"].astype(str) != str(film_id)]
    dropped = before_count - len(master_df)
    if dropped:
        print(f"  Replaced {dropped} existing rows for '{film_id}'")
else:
    master_df = pd.DataFrame(columns=film_characters_df.columns)

updated_master = pd.concat(
    [master_df, film_characters_df.astype(str)],
    ignore_index=True,
)
updated_master.to_csv(MASTER_CHARS, index=False)

print(f"Master characters.csv written to: {MASTER_CHARS}")
print(f"  Total films in master  : {updated_master['film_id'].nunique()}")
print(f"  Total characters       : {len(updated_master)}")

  Replaced 49 existing rows for 'elemental_2023'
Master characters.csv written to: C:\Users\majoa\OneDrive\Desktop\Learning\Grad_School\final_master_thesis\CLEAN\data\metadata\characters.csv
  Total films in master  : 18
  Total characters       : 994


In [16]:
# Show a per-film breakdown of what is currently in the master file
summary = (
    updated_master
    .groupby("film_id", sort=False)
    .agg(
        n_characters=("character_id", "count"),
        n_female=("gender", lambda x: (x == "F").sum()),
        n_male=("gender", lambda x: (x == "M").sum()),
        n_network=("included_in_network", lambda x: (x.str.lower().isin({"true"})).sum()),
        protagonist=("canonical_name", lambda x: ", ".join(
            updated_master.loc[
                (updated_master["film_id"] == x.iloc[0] if len(x) else "") &
                (updated_master["is_protagonist"].str.lower() == "true"),
                "canonical_name",
            ].tolist() if len(x) else []
        )),
    )
    .reset_index()
)

# Simpler protagonist lookup
protagonists = (
    updated_master[updated_master["is_protagonist"].str.lower() == "true"]
    .groupby("film_id")["canonical_name"]
    .apply(lambda x: ", ".join(x))
    .reset_index()
    .rename(columns={"canonical_name": "protagonist"})
)
summary = (
    updated_master
    .groupby("film_id", sort=False)
    .agg(
        n_characters=("character_id", "count"),
        n_female=("gender", lambda x: (x == "F").sum()),
        n_male=("gender", lambda x: (x == "M").sum()),
        n_network=("included_in_network", lambda x: (x.str.lower() == "true").sum()),
    )
    .reset_index()
    .merge(protagonists, on="film_id", how="left")
)

print("Master characters.csv — current state")
print(f"({len(updated_master)} total rows across {updated_master['film_id'].nunique()} films)\n")
display(summary)

Master characters.csv — current state
(994 total rows across 18 films)



,film_id,n_characters,n_female,n_male,n_network,protagonist
0,beautyandthebeast_1991,59,14,39,14,Belle
1,cars2,59,7,44,24,Mater
2,coco_2017,76,21,42,22,"Miguel, H Ctor"
3,encanto_2021,45,21,10,15,Mirabel
4,findingnemo,70,9,39,25,Marlin
5,frozen_2013,53,11,23,11,Anna
6,incredibles_2_2018,54,13,34,16,"Bob, Elastigirl"
7,inside_out_2015,68,31,33,19,"Joy, Riley"
8,mulan_1998,32,9,16,13,Mulan
9,onward_2020,46,15,28,14,Ian
